In [1]:
import os
import time
from itertools import product

import numpy as np
import pandas as pd
import gurobipy as gb
from sklearn.linear_model import LinearRegression

# -----------------------------
# Setup and Data Loading
# -----------------------------

# WLS credentials
WLS_ACCESS_ID = 'ccc2c36a-db14-4956-b2e3-60adc45e9957'
WLS_SECRET = '1e0e3dbf-7933-44dc-8f81-e0482ded7ac8'
LICENSE_ID = 2586688

# Create the Gurobi environment with parameters
env = gb.Env(empty=True)
env.setParam('WLSACCESSID', WLS_ACCESS_ID)
env.setParam('WLSSECRET', WLS_SECRET)
env.setParam('LICENSEID', LICENSE_ID)
env.start()

# Load data and neighborhood matrices
df = pd.read_csv('GA_features.csv')
NEIGHBOR_INDEX_MATRIX = np.load('index_matrix.npy')
NEIGHBOR_DISTANCE_MATRIX = np.load('distance_matrix.npy')

# -----------------------------
# Constants and Columns
# -----------------------------
SOCIAL_CATEGORIES = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
TAU_TIGHTEST = 0.4  # Tightest fairness constraint
TAU_NONE = None   # No fairness constraint

FEATURE_COLUMNS = ['frac_unem', 'n_poll', 'contribution', 'tweets']
COUNT_COLUMNS = [f'registered_{cat}' for cat in SOCIAL_CATEGORIES]
FRAC_COLUMNS = [f'frac_registered_{cat}' for cat in SOCIAL_CATEGORIES]

# -----------------------------
# Prepare Features and Targets
# -----------------------------
X = df[FEATURE_COLUMNS]
A_frac = df[FRAC_COLUMNS]
y_train = df['frac_votes'].values

# Feature values
UNEMPLOYMENT_RATE = X['frac_unem'].values
POLLING_STATIONS = X['n_poll'].values
# Use a constant vector for contribution as in the original code
CONTRIBUTION = np.ones_like(X['contribution'].values)
TWEETS = X['tweets'].values

# Neighborhood dimensions and intervention space
NUM_SCHOOLS = X.shape[0]
NUM_NEIGHBORS = NEIGHBOR_INDEX_MATRIX.shape[1]
INTERVENTION_SAMPLE_SPACES = [(0, 1)] * NUM_NEIGHBORS
POSSIBLE_INTERVENTIONS_MATRIX = np.array(list(product(*INTERVENTION_SAMPLE_SPACES)))
NUM_POSSIBLE_INTERVENTIONS = POSSIBLE_INTERVENTIONS_MATRIX.shape[0]

# -----------------------------
# Regression Model Setup
# -----------------------------
def compute_adjusted_features(feature_values, A_frac, neighbor_distance_matrix):
    # Compute the maximum influence from neighbors and multiply elementwise with A_frac
    max_neighbor_influence = np.max(neighbor_distance_matrix * feature_values[:, None], axis=1).reshape(NUM_SCHOOLS, 1)
    return A_frac * max_neighbor_influence

# Compute adjusted features for each feature
adjusted_unemployment = compute_adjusted_features(UNEMPLOYMENT_RATE, A_frac, NEIGHBOR_DISTANCE_MATRIX)
adjusted_polling = compute_adjusted_features(POLLING_STATIONS, A_frac, NEIGHBOR_DISTANCE_MATRIX)
adjusted_contribution = compute_adjusted_features(CONTRIBUTION, A_frac, NEIGHBOR_DISTANCE_MATRIX)
adjusted_tweets = compute_adjusted_features(TWEETS, A_frac, NEIGHBOR_DISTANCE_MATRIX)

# Combine features and train regression model (no intercept)
X_train = np.concatenate((adjusted_unemployment, adjusted_polling, adjusted_contribution, adjusted_tweets, A_frac), axis=1)
linear_model = LinearRegression(fit_intercept=False).fit(X_train, y_train)
model_weights = linear_model.coef_
param_dims = len(SOCIAL_CATEGORIES)

# Split regression weights into groups corresponding to each feature and demographics
weight_dict = {
    'alpha': model_weights[:param_dims],
    'beta': model_weights[param_dims:2*param_dims],
    'gamma': model_weights[2*param_dims:3*param_dims],
    'delta': model_weights[3*param_dims:4*param_dims],
    'theta': model_weights[4*param_dims:]
}
params = pd.DataFrame(weight_dict, index=SOCIAL_CATEGORIES)
ALPHA = params['alpha'].values
BETA = params['beta'].values
GAMMA = params['gamma'].values
DELTA = params['delta'].values
THETA = params['theta'].values

# -----------------------------
# Impact Calculation Functions
# -----------------------------
def calculate_expected_impact(index, intervention_array, demographic_vector):
    """
    Calculate the expected impact for a single school using the intervention decision.
    For each term, we multiply the (binary) intervention vector by the neighbor distances,
    take the maximum value, and weight it by the corresponding regression coefficient.
    """
    # Get the indices and distances of the school's neighbors
    nearest_neighbors = NEIGHBOR_INDEX_MATRIX[index, :]
    neighbor_distances = NEIGHBOR_DISTANCE_MATRIX[index, nearest_neighbors]
    
    unemployment_term = np.dot(demographic_vector, ALPHA) * np.max(neighbor_distances * intervention_array)
    polling_term = np.dot(demographic_vector, BETA) * np.max(neighbor_distances * POLLING_STATIONS[nearest_neighbors])
    contribution_term = np.dot(demographic_vector, GAMMA) * np.max(neighbor_distances * CONTRIBUTION[nearest_neighbors])
    tweets_term = np.dot(demographic_vector, DELTA) * np.max(neighbor_distances * TWEETS[nearest_neighbors])
    demographic_term = np.dot(demographic_vector, THETA)
    
    impact = unemployment_term + polling_term + contribution_term + tweets_term + demographic_term
    return max(min(impact, 1), 0)

def calculate_all_possible_impacts(index, demographic_vector):
    """
    Compute the expected impact for all possible neighbor intervention patterns.
    """
    possible_impacts = np.empty(NUM_POSSIBLE_INTERVENTIONS)
    for k, intervention_array in enumerate(POSSIBLE_INTERVENTIONS_MATRIX):
        possible_impacts[k] = calculate_expected_impact(index, intervention_array, demographic_vector)
    return possible_impacts

def calculate_total_impact(intervention_array):
    """
    Calculate the total impact over all schools given a binary intervention solution.
    """
    total_impact = 0
    for i in range(NUM_SCHOOLS):
        demographic_vector = A_frac.values[i, :]
        total_impact += calculate_expected_impact(i, intervention_array[i], demographic_vector)
    return total_impact

# -----------------------------
# Optimization Routine
# -----------------------------
def optimize_interventions(tau_value, budget):
    """
    Optimize the intervention decisions for all schools under a budget constraint.
    A fairness constraint (tau) may be imposed. For each school the model selects
    an intervention pattern (from a discrete set) that yields a factual impact,
    while the fairness constraints force the differences in impact (for each demographic group)
    to be within tau.
    """
    print(f'Running optimization for tau={tau_value} and budget={budget}')
    model = gb.Model(env=env)
    
    # Binary variables for each school
    interventions = model.addVars(NUM_SCHOOLS, vtype=gb.GRB.BINARY, name="interventions")
    model.addConstr(sum(interventions[i] for i in range(NUM_SCHOOLS)) <= budget, "budget_constraint")
    
    # For each school, create auxiliary variables for each possible intervention pattern
    for index in range(NUM_SCHOOLS):
        demographic_vector = A_frac.values[index, :]
        factual_impacts = calculate_all_possible_impacts(index, demographic_vector)
        auxiliary_vars = model.addVars(len(factual_impacts), obj=factual_impacts, vtype=gb.GRB.CONTINUOUS, name=f"aux_{index}")
        model.update()
        
        # Link the auxiliary variables to the intervention decisions of the neighbors
        for j, intervention_pattern in enumerate(POSSIBLE_INTERVENTIONS_MATRIX):
            for k, neighbor in enumerate(NEIGHBOR_INDEX_MATRIX[index, :]):
                if intervention_pattern[k] == 1:
                    model.addConstr(auxiliary_vars[j] <= interventions[int(neighbor)])
                else:
                    model.addConstr(auxiliary_vars[j] <= 1 - interventions[int(neighbor)])
        model.addConstr(sum(auxiliary_vars[j] for j in range(len(factual_impacts))) == 1)
        
        # If a fairness constraint is imposed, add constraints on group impact differences
        if tau_value is not None:
            for group_idx in range(A_frac.shape[1]):
                group_indicator = np.eye(A_frac.shape[1])[group_idx]
                group_impact_diff = calculate_all_possible_impacts(index, group_indicator) - factual_impacts
                model.addConstr(
                    sum(auxiliary_vars[j] * group_impact_diff[j] for j in range(len(factual_impacts))) <= tau_value
                )
    
    model.setObjective(model.getObjective(), gb.GRB.MAXIMIZE)
    model.optimize()
    
    if model.status == gb.GRB.OPTIMAL:
        solution = np.array([interventions[i].X for i in range(NUM_SCHOOLS)]).astype(bool)
        return solution
    else:
        raise RuntimeError("Optimization failed.")

# -----------------------------
# Comparison Routine
# -----------------------------
def compare_solutions(budget):
    """
    Run optimization for a given budget using the tight tau and no tau,
    then compare the solutions and print the corresponding impacts.
    """
    print(f"\nComparing solutions for budget = {budget}")
    try:
        tight_solution = optimize_interventions(TAU_TIGHTEST, budget)
        tight_impact = calculate_total_impact(tight_solution)
    except Exception as e:
        print(f"Optimization failed for TAU_TIGHTEST with budget {budget}: {e}")
        return None, None, None
    
    try:
        no_tau_solution = optimize_interventions(TAU_NONE, budget)
        no_tau_impact = calculate_total_impact(no_tau_solution)
    except Exception as e:
        print(f"Optimization failed for TAU_NONE with budget {budget}: {e}")
        return None, None, None
    
    print(f"Tight constraint (tau={TAU_TIGHTEST}) solution total impact: {tight_impact}")
    print(f"No tau solution total impact: {no_tau_impact}")
    different = not np.array_equal(tight_solution, no_tau_solution)
    if different:
        print("The solutions are different.")
    else:
        print("The solutions are the same.")
    return tight_solution, no_tau_solution, different

# -----------------------------
# Main Loop: Finding a Budget with Different Solutions
# -----------------------------
# -----------------------------
# Main Loop: Finding a Budget with Different Solutions
# -----------------------------
SAVE_DIR = "Results/"
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

BUDGETS = list(range(41, 70))  # List of budgets from 1 to 40
MIN_DIFFERENCES = 5  # Minimum number of differences required to save results

found_different = False
for budget in BUDGETS:
    print("=" * 80)
    print(f"Running optimization for budget = {budget}")
    
    # Run optimization for tight tau and no tau
    tight_sol, no_tau_sol, is_different = compare_solutions(budget)
    if is_different is None:
        continue  # Skip if optimization failed
    
    # Check if the solutions have at least MIN_DIFFERENCES differences
    if is_different:
        num_differences = np.sum(tight_sol != no_tau_sol)
        print(f"Number of differences between solutions: {num_differences}")
        
        if num_differences >= MIN_DIFFERENCES:
            print(f"Solutions have {num_differences} differences (>= {MIN_DIFFERENCES}). Saving results.")
            
            # Save results to files
            tight_file = os.path.join(SAVE_DIR, f"tight_tau_budget_{budget}.npy")
            no_tau_file = os.path.join(SAVE_DIR, f"no_tau_budget_{budget}.npy")
            np.save(tight_file, tight_sol)
            np.save(no_tau_file, no_tau_sol)
            print(f"Saved tight constraint solution to {tight_file}")
            print(f"Saved no tau solution to {no_tau_file}")
            found_different = True
        else:
            print(f"Solutions have {num_differences} differences (< {MIN_DIFFERENCES}). Not saving results.")
    else:
        print("The solutions are the same. Not saving results.")

if not found_different:
    print("\nNo budget found where the solutions differ by at least 5 interventions.")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2586688
Academic license 2586688 - for non-commercial use only - registered to ru___@ucsd.edu
Running optimization for budget = 41

Comparing solutions for budget = 41
Running optimization for tau=0.4 and budget=41
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (linux64 - "Ubuntu 22.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU E5-2630 v4 @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 20 physical cores, 40 logical processors, using up to 20 threads

Academic license 2586688 - for non-commercial use only - registered to ru___@ucsd.edu
Optimize a model with 62329 rows, 10335 columns and 198975 nonzeros
Model fingerprint: 0x0de36001
Variable types: 10176 continuous, 159 integer (159 binary)
Coefficient statistics:
  Matrix range     [3e-05, 1e+00]
  Objective range  [1e-02, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [4e-01, 4e+01]
Presolve removed 18034 rows and 2516 columns
Preso

In [1]:
import pandas as pd 
import numpy as np

In [2]:
df = pd.read_csv('GA_features.csv')
df

,Unnamed: 0.2,Unnamed: 0,Unnamed: 0.1,county,tweets,contribution,n_poll,frac_unem,frac_votes,total_votes,...,frac_voted_A,frac_voted_B,frac_voted_C,frac_voted_D,frac_voted_E,frac_voted_F,frac_voted_G,total_registers,latitude,longitude
0,0,0,0,APPLING,0,235818,0,0.160193,0.695728,8403,...,0.155540,0.783292,0.004165,0.012972,0.001071,0.004046,0.038915,12078,31.7492,-82.2889
1,1,1,1,ATKINSON,0,93026,0,0.117003,0.649241,3167,...,0.187243,0.729081,0.000947,0.050837,0.000947,0.003789,0.027155,4878,31.2971,-82.8800
2,2,2,2,BACON,0,173074,0,0.115665,0.672325,4674,...,0.085794,0.858151,0.002353,0.010270,0.000642,0.003637,0.039153,6952,31.5537,-82.4527
3,3,3,3,BAKER,0,72226,0,0.179682,0.682852,1561,...,0.383088,0.570788,0.006406,0.006406,0.001281,0.002562,0.029468,2286,31.3262,-84.4447
4,4,4,4,BALDWIN,2,927608,0,0.173954,0.673200,18342,...,0.374932,0.572239,0.008123,0.007796,0.000872,0.005670,0.030367,27246,33.0693,-83.2496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,154,154,154,WHITFIELD,0,4216384,1,0.070532,0.660559,36955,...,0.034745,0.749127,0.009552,0.127588,0.001380,0.009796,0.067812,55945,34.8056,-84.9672
155,155,155,155,WILCOX,0,101176,0,0.182849,0.690566,3294,...,0.230419,0.740437,0.002429,0.007286,0.000607,0.003643,0.015179,4770,31.9729,-83.4323
156,156,156,156,WILKES,0,142526,0,0.206583,0.703740,5024,...,0.344944,0.605494,0.004976,0.004379,0.001393,0.005772,0.033041,7139,33.7819,-82.7432
157,157,157,157,WILKINSON,0,92444,0,0.163246,0.711561,4776,...,0.383585,0.581449,0.001675,0.002094,0.001466,0.003769,0.025963,6712,32.8024,-83.1712


In [4]:
# List of counties to set n_poll = 1
target_counties = [
    'BAKER', 'CALHOUN', 'CHARLTON', 'CLAY', 'CLINCH', 'DODGE', 'EARLY', 'GLASCOCK',
    'HANCOCK', 'LINCOLN', 'MONTGOMERY', 'QUITMAN', 'RANDOLPH', 'SCREVEN', 'STEWART',
    'TALBOT', 'TALIAFERRO', 'WILCOX', 'WILKINSON', 'WORTH'
]

# Update n_poll: 1 for target counties, 0 for others
df['n_poll'] = df['county'].apply(lambda x: 1 if x in target_counties else 0)

# Display the modified DataFrame
df

,Unnamed: 0.2,Unnamed: 0,Unnamed: 0.1,county,tweets,contribution,n_poll,frac_unem,frac_votes,total_votes,...,frac_voted_A,frac_voted_B,frac_voted_C,frac_voted_D,frac_voted_E,frac_voted_F,frac_voted_G,total_registers,latitude,longitude
0,0,0,0,APPLING,0,235818,0,0.160193,0.695728,8403,...,0.155540,0.783292,0.004165,0.012972,0.001071,0.004046,0.038915,12078,31.7492,-82.2889
1,1,1,1,ATKINSON,0,93026,0,0.117003,0.649241,3167,...,0.187243,0.729081,0.000947,0.050837,0.000947,0.003789,0.027155,4878,31.2971,-82.8800
2,2,2,2,BACON,0,173074,0,0.115665,0.672325,4674,...,0.085794,0.858151,0.002353,0.010270,0.000642,0.003637,0.039153,6952,31.5537,-82.4527
3,3,3,3,BAKER,0,72226,1,0.179682,0.682852,1561,...,0.383088,0.570788,0.006406,0.006406,0.001281,0.002562,0.029468,2286,31.3262,-84.4447
4,4,4,4,BALDWIN,2,927608,0,0.173954,0.673200,18342,...,0.374932,0.572239,0.008123,0.007796,0.000872,0.005670,0.030367,27246,33.0693,-83.2496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,154,154,154,WHITFIELD,0,4216384,0,0.070532,0.660559,36955,...,0.034745,0.749127,0.009552,0.127588,0.001380,0.009796,0.067812,55945,34.8056,-84.9672
155,155,155,155,WILCOX,0,101176,1,0.182849,0.690566,3294,...,0.230419,0.740437,0.002429,0.007286,0.000607,0.003643,0.015179,4770,31.9729,-83.4323
156,156,156,156,WILKES,0,142526,0,0.206583,0.703740,5024,...,0.344944,0.605494,0.004976,0.004379,0.001393,0.005772,0.033041,7139,33.7819,-82.7432
157,157,157,157,WILKINSON,0,92444,1,0.163246,0.711561,4776,...,0.383585,0.581449,0.001675,0.002094,0.001466,0.003769,0.025963,6712,32.8024,-83.1712


In [5]:
df.to_csv('GA_features.csv')